# Trader Performance vs Market Sentiment Analysis
Data Science Intern Assignment


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Load Data

In [ ]:

fear = pd.read_csv('fear_greed_index.csv')
trades = pd.read_csv('historical_data.csv')

print("Fear/Greed shape:", fear.shape)
print("Trades shape:", trades.shape)

fear.head(), trades.head()


## Data Cleaning

In [ ]:

# Convert timestamps
trades['date'] = pd.to_datetime(trades['Timestamp'], unit='ms', errors='coerce').dt.date
fear['date'] = pd.to_datetime(fear['date']).dt.date

# Missing values
print("Missing values trades:")
print(trades.isna().sum())

print("Missing values sentiment:")
print(fear.isna().sum())

# Remove duplicates
trades = trades.drop_duplicates()
fear = fear.drop_duplicates()


## Feature Engineering

In [ ]:

# Win / loss
trades['win'] = trades['Closed PnL'] > 0

daily = trades.groupby(['Account','date']).agg(
    daily_pnl=('Closed PnL','sum'),
    trades_per_day=('Trade ID','count'),
    avg_trade_size=('Size USD','mean')
).reset_index()

win_rate = trades.groupby(['Account','date'])['win'].mean().reset_index(name='win_rate')

daily = daily.merge(win_rate, on=['Account','date'], how='left')

# Merge sentiment
merged = daily.merge(fear[['date','classification']], on='date', how='left')

merged.head()


## Sentiment vs Performance

In [ ]:

summary = merged.groupby('classification').agg(
    avg_daily_pnl=('daily_pnl','mean'),
    avg_win_rate=('win_rate','mean'),
    avg_trades=('trades_per_day','mean'),
    avg_trade_size=('avg_trade_size','mean'),
    observations=('daily_pnl','count')
).sort_values('observations',ascending=False)

print(summary)


## Visualization

In [ ]:

summary[['avg_daily_pnl']].plot(kind='bar')
plt.title("Average Daily PnL by Market Sentiment")
plt.ylabel("PnL")
plt.show()

summary[['avg_win_rate']].plot(kind='bar')
plt.title("Win Rate by Market Sentiment")
plt.ylabel("Win Rate")
plt.show()



## Example Insights

1. Trader profitability changes across sentiment regimes.
2. Fear periods often show different trading intensity than Greed periods.
3. Trade frequency and win rate vary with sentiment.

## Strategy Ideas

1. Reduce leverage during Extreme Greed periods due to potential volatility.
2. Increase trade participation during Fear periods when market reversals occur.
